On a tout d'abord la création d'une arborescence récursive qui nous permettra de tester notre crawler. Cette arborescence contient des fichiers de taille différentes imbriqués dans des dossiers.

In [ ]:
import random
import string
import lorem
from pathlib import Path

def random_folder_name(length=8):
    """Génère un nom de dossier aléatoire."""
    return ''.join(random.choices(string.ascii_lowercase, k=length))

def random_file_name(length=8):
    """Génère un nom de fichier aléatoire."""
    return ''.join(random.choices(string.ascii_lowercase, k=length)) + ".txt"

def create_random_tree(
    root,
    max_depth=3,
    max_subfolders=4,
    max_files=20,
    lorem_min=1,
    lorem_max=10
):
    """
    Génère récursivement une arborescence de dossiers et fichiers :
    - max_depth : profondeur maximale d'imbrication
    - max_subfolders : nombre max de sous-dossiers par dossier
    - max_files : nombre max de fichiers par dossier
    - lorem_min/lorem_max : nombre de paragraphes lorem ipsum aléatoires par fichier
    """

    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)

    ## Création des fichiers dans le dossier courant d'un nombre aléatoire allant de 0 à max_files
    num_files = random.randint(0, max_files)
    for _ in range(num_files):
        file_path = root / random_file_name()
        with open(file_path, "w", encoding="utf-8") as f:
            # Génération de contenu lorem ipsum avec un nombre aléatoire de paragraphes 
            # pour faire varier la taille des fichiers
            paragraphs = random.randint(lorem_min, lorem_max)
            content = "\n\n".join(lorem.paragraph() for _ in range(paragraphs))
            f.write(content)

    if max_depth <= 0:
        return
    
    ## Création des sous-dossiers imbriqués dans les dossiers parents d'un nombre aléatoire
    ## allant de 0 à max_subfolders
    num_subfolders = random.randint(0, max_subfolders)
    for _ in range(num_subfolders):
        subfolder = root / random_folder_name()
        create_random_tree(
            subfolder,
            max_depth=max_depth - 1,
            max_subfolders=max_subfolders,
            max_files=max_files,
            lorem_min=lorem_min,
            lorem_max=lorem_max
        )

if __name__ == "__main__":
    base_directory = "test_arbo"

    max_depth = 6
    max_subfolders = 10
    max_files = 50
    lorem_min = 1
    lorem_max = 3

    print("Génération de l'arborescence en cours...")
    
    create_random_tree(
        root=base_directory,
        max_depth=max_depth,
        max_subfolders=max_subfolders,
        max_files=max_files,
        lorem_min=lorem_min,
        lorem_max=lorem_max
    )

    print(f"Arborescence générée dans le dossier : '{base_directory}'")


Maintenant, on crée un algorithme de test avec la méthode os.walk. L'objectif va être d'aller compter le nombre de paragraphes de lorem ipsum pour ressortir les fichiers qui n'ont que 3 paragraphes et on regarde le temps que cela mets.

In [ ]:
import os
import time

start_dir = "test_arbo/"
found_paths = []

start_timer = time.time()

def count_paragraphs(file_path):
    """Compte les paragraphes séparés par au moins une ligne vide."""
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()

    # On sépare sur une ou plusieurs lignes vides
    paragraphs = [p.strip() for p in content.split("\n\n") if p.strip()]
    return len(paragraphs)

for root, dirs, files in os.walk(start_dir):
    for file in files:
        full_path = os.path.join(root, file)

        # On ne traite que les .txt
        if file.lower().endswith(".txt"):
            try:
                if count_paragraphs(full_path) == 3:
                    found_paths.append(full_path)
            except Exception as e:
                print(f"[ERREUR] Impossible de lire {full_path} : {e}")

# Résultats
print("\n--------------------------------------")
if found_paths:
    print(f"--> {len(found_paths)} fichier(s) avec exactement 3 paragraphes :")
    for p in found_paths:
        print(" -", p)
else:
    print("Aucun fichier avec exactement 3 paragraphes trouvé.")

end_time = time.time() - start_timer
print(f"\nTemps d'exécution : {end_time:.4f} secondes")


Maintenant, quelques graphiques représentants les performances de nos crawlers

In [ ]:
import matplotlib.pyplot as plt

# Données
threads = list(range(2, 20))
osWalk = 2.137456413333333
temps_sans_vol = [
    2.805856466293335, 2.0422867139180503, 1.8359113534291585, 2.3634514808654785,
    2.0915769735972085, 2.103187163670858, 2.071011702219645, 2.0119806925455728,
    2.6506969134012857, 2.5903857549031577, 2.326402425765991, 2.749156395594279,
    3.243384838104248, 2.3048545519510903, 2.340400139490763, 2.3440900643666587,
    3.0054310957590737, 2.487187385559082
]

temps_avec_vol = [
    2.0990055402119956, 1.67915145556132, 1.567650318145752, 1.669169267018636,
    1.7513261636098225, 2.0222861766815186, 2.217916170756022, 2.331177075703939,
    2.829897880554199, 3.043776591618856, 2.9092787901560464, 2.7448483308156333,
    2.908443053563436, 2.9944852193196616, 2.9120771884918213, 3.001306931177775,
    2.917454719543457, 3.0554562409718833
]

# Création du graphique
plt.figure(figsize=(10, 6))
plt.plot(threads, [osWalk]*len(threads), linestyle='--', color='green', label='os.walk()')
plt.plot(threads, temps_sans_vol, marker='o', label='Sans vol de pile', color='blue')
plt.plot(threads, temps_avec_vol, marker='o', label='Avec vol de pile', color='red')
plt.xlabel("Nombre de threads")
plt.ylabel("Temps d'exécution (s)")
plt.title("Performance selon le nombre de threads et le vol de pile")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.xticks(threads)  # affiche tous les threads sur l'axe X
plt.tight_layout()
plt.show()


In [ ]:
# Données
threads = list(range(2, 20))

temps_threads_vol = [
    2.0990055402119956, 1.67915145556132, 1.567650318145752, 1.669169267018636,
    1.7513261636098225, 2.0222861766815186, 2.217916170756022, 2.331177075703939,
    2.829897880554199, 3.043776591618856, 2.9092787901560464, 2.7448483308156333,
    2.908443053563436, 2.9944852193196616, 2.9120771884918213, 3.001306931177775,
    2.917454719543457, 3.0554562409718833
]

temps_process_work_steal = [
    10.245505491892496, 6.8437105019887285, 6.113382895787557, 6.016989787419637,
    5.962341864903768, 6.377739508946736, 5.971661488215129, 6.370392243067424,
    6.572675546010335, 6.317737261454265, 6.942087650299072, 7.47414763768514,
    7.492346366246541, 7.637696822484334, 7.423251628875732, 7.383508205413818,
    7.860464572906494, 7.943342526753743
]

# Création du graphique
plt.figure(figsize=(10, 6))
plt.plot(threads, temps_threads_vol, marker='o', label='Threads avec vol de pile', color='blue')
plt.plot(threads, temps_process_work_steal, marker='o', label='Process work steal', color='green')
plt.xlabel("Nombre de threads")
plt.ylabel("Temps d'exécution (s)")
plt.title("Comparaison Threads avec vol de pile vs Process Work Steal")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.xticks(threads)
plt.tight_layout()
plt.show()
